# Test MLP Deep


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [ ]:
DATA_DIR = Path("data")
MODEL_DIR = Path("models")
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

TEST_PATH = DATA_DIR / "test_features-1.csv" 
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor_v2.joblib"
TARGET_SCALER_PATH = MODEL_DIR / "target_scaler_v2.joblib"
MODEL_PATH = MODEL_DIR / "best_mlp_v2.pt"
RIDGE_PATH = MODEL_DIR / "best_baseline_v2.joblib"

required = [PREPROCESSOR_PATH, TARGET_SCALER_PATH, MODEL_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Falta el archivo requerido: {p}")

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro {TEST_PATH}. Guarda el test dentro de data/ o cambia TEST_PATH al nombre correcto."
    )

print("Test:", TEST_PATH)
print("Modelo Deep:", MODEL_PATH)


Test: data/test_features-1.csv
Modelo Deep: models/best_mlp_v2.pt


## Cargar el test


In [3]:
test_df = pd.read_csv(TEST_PATH)
print("Shape:", test_df.shape)
display(test_df.head())

HAS_TARGET = "SalePrice" in test_df.columns
HAS_ID = "Id" in test_df.columns
print("¿Incluye SalePrice?:", HAS_TARGET)
print("¿Incluye Id?:", HAS_ID)


Shape: (292, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,893,20,RL,70.0,8414,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal
1,1106,60,RL,98.0,12256,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
2,414,30,RM,56.0,8960,Pave,Grvl,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal
3,523,50,RM,50.0,5000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal
4,1037,20,RL,89.0,12898,Pave,NaN,IR1,HLS,AllPub,...,0,0,NaN,NaN,NaN,0,9,2009,WD,Normal


¿Incluye SalePrice?: False
¿Incluye Id?: True


## Separar variables y aplicar el preprocesamiento 


In [4]:
ids = test_df["Id"].copy() if HAS_ID else pd.Series(np.arange(len(test_df)), name="Id")
y_test_usd = test_df["SalePrice"].to_numpy(dtype=np.float32) if HAS_TARGET else None

cols_to_drop = [c for c in ["SalePrice", "Id"] if c in test_df.columns]
X_test_raw = test_df.drop(columns=cols_to_drop)

preprocessor = joblib.load(PREPROCESSOR_PATH)
X_test = preprocessor.transform(X_test_raw)
if hasattr(X_test, "toarray"):
    X_test = X_test.toarray()
X_test = np.asarray(X_test, dtype=np.float32)

print("X test procesado:", X_test.shape)
print("NaN:", np.isnan(X_test).sum())
assert np.isfinite(X_test).all(), "Hay valores no finitos después del preprocesamiento."


X test procesado: (292, 278)
NaN: 0


## Cargar MLP Deep entrenada


In [5]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
target_scaler = joblib.load(TARGET_SCALER_PATH)
Y_MEAN = float(target_scaler["mean"])
Y_STD = float(target_scaler["std"])

print("Arquitectura guardada:", checkpoint["architecture_name"], checkpoint["hidden_layers"])
print("Hiperparámetros:", checkpoint["hyperparams"])
print(f"RMSE de validación guardado: ${checkpoint['val_rmse_usd']:,.2f}")

assert checkpoint["architecture_name"] == "deep", "El checkpoint cargado no es Medium."
assert list(checkpoint["hidden_layers"]) == [256, 128, 64, 32], "La arquitectura no es [256, 128, 64, 32]."


Arquitectura guardada: deep [256, 128, 64, 32]
Hiperparámetros: {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0001, 'batch_size': 32, 'max_epochs': 600, 'patience': 50}
RMSE de validación guardado: $22,500.09


In [6]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

hidden = checkpoint["hidden_layers"]
dropout = checkpoint["hyperparams"]["dropout"]
model = MLP(X_test.shape[1], hidden, dropout).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Modelo Deep cargado correctamente.")


Modelo Deep cargado correctamente.


## Generar predicciones


In [7]:
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    pred_scaled = model(X_test_t).cpu().numpy()

pred_usd = pred_scaled * Y_STD + Y_MEAN

pred_df = pd.DataFrame({
    "Id": ids.to_numpy(),
    "Prediction": pred_usd
})

display(pred_df.head())
print(f"Predicción mínima: ${pred_usd.min():,.2f}")
print(f"Predicción media:  ${pred_usd.mean():,.2f}")
print(f"Predicción máxima: ${pred_usd.max():,.2f}")


,Id,Prediction
0,893,156226.343750
1,1106,358271.750000
2,414,98513.195312
3,523,168959.921875
4,1037,352107.843750


Predicción mínima: $39,661.16
Predicción media:  $179,921.03
Predicción máxima: $541,693.31


## Guardar predicciones


In [8]:
OUTPUT_PATH = REPORT_DIR / "predicciones_test_deep.csv"
pred_df.to_csv(OUTPUT_PATH, index=False)
print("Guardado:", OUTPUT_PATH)


Guardado: reports/predicciones_test_deep.csv
